# MCC labels → DRP (без повторной загрузки trx)

Уже залитая таблица `sbx_da.tmp_shestopalov_acq_mcc_month` читается из DRP,
к ней добавляются `mcc_name_ru` / `mcc_label`, таблица **перезаписывается**.

**Impala / помесячный fetch не нужен.**

После прогона в Superset: Dataset → Sync columns → в Table dimension = `mcc_label`.


In [ ]:
import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

# Захардкоженный справочник (из mcc.txt + русские названия)
MCC_NAME_RU = {
  "149": "Прочие (код 149 — уточнить)",
  "1520": "Генеральные подрядчики - жилищное и торговое строительство",
  "1711": "Генеральные подрядчики по вентиляции, теплоснабжению, и водопроводу",
  "1731": "Подрядчики по электричеству",
  "1799": "Подрядчики, специализированная торговля - нигде более не классифицированные",
  "2741": "Разнообразные издательства/ печатное дело",
  "3543": "Отели Four Seasons",
  "4111": "Транспортировка - пригородные и локальные сезонные транспортные средства, включая электрички",
  "4112": "Пассажирские железнодорожные перевозки",
  "4131": "Автобусные линии, туристические автобусы",
  "4214": "Агентства по автотранспортным перевозкам, местные/дальные автогрузоперевоз-ки, компании попереезду и хранению, местная доставка",
  "4215": "Услуги курьера - по воздуху и на земле, агентство по отправке грузов",
  "4722": "Туроператоры, туристические агентства и организаторы экскурсий",
  "4812": "Магазины телекоммуникационного оборудования (телефоны, факсы, пейджеры и др.)",
  "4814": "Телекоммуникационные услуги, включая, но не ограничивая предоплаченые телефонные услуги и периодические телефонные услуги",
  "4816": "Компьютерная сеть/ информационные услуги",
  "4899": "Кабельные и другие платные телевизионные услуги",
  "4900": "Коммунальные услуги - электричество, газ, санитария, вода",
  "5021": "Офисная и торговая мебель",
  "5047": "Стоматологичееское/лабороторное/медицинское/офтальмологическое стацоинарное оборудование и устройства",
  "5051": "Центры и офисы работ по металлу",
  "5074": "Оборудование для водопровода и отопительной системы",
  "5169": "Химикалии и смежные вещества , не классифицированные ранее",
  "5192": "Книги, периодические издания и газеты",
  "5200": "Магазины бытовой техники",
  "5211": "Лесо- и строительный материал",
  "5231": "Розничная продажа стекла, красок и обоев",
  "5251": "Скобяные товары в розницу",
  "5261": "Садовые принадлежности (в том числе для ухода за газонами) в розницу",
  "5300": "Оптовики",
  "5309": "Беспошлинные магазины duty free",
  "5331": "Универсальные магазины",
  "5399": "Различные товары общего назначения",
  "5411": "Продуктовые магазины, супермаркеты",
  "5422": "Продажа свежего и мороженого мяса",
  "5441": "Кондитерские",
  "5451": "Продажа молочных продуктов в розницу",
  "5462": "Булочные, пекарни",
  "5499": "Различные продовольственные магазины - рынки, магазины со спец. ассортиментом, продажа полуфабрикатов, фирменных блюд, продажа с помощью торговых автоматов",
  "5511": "Легковой и грузовой транспорт - продажа, сервис, ремонт, запчасти и лизинг",
  "5533": "Запчасти, автомобильные аксессуары",
  "5541": "Бензиновые станции обслуживания (с или без дополнительного сервиса)",
  "5571": "Продажа мотоциклов",
  "5599": "Автодиллеры",
  "5611": "Мужская и подростковая одежда и аксессуары",
  "5621": "Магазины женской одежды",
  "5631": "Женские магазины аксессуаров",
  "5641": "Магазины детской одежды",
  "5651": "Семейные магазины одежды",
  "5655": "Спортивная одежда, одежда для верховой езды и езды на мотоцикле",
  "5661": "Магазины обуви",
  "5681": "Изготовление и продажа меховых изделий",
  "5691": "Магазины мужской и женской одежды",
  "5697": "Ателье",
  "5699": "Магазины одежды и аксессуаров",
  "5712": "Оборудование, мебель и бытовые принадлежности (кроме электрооборудования)",
  "5713": "Покрытия для пола",
  "5714": "Ткани, обивочный материал, гардины и портьеры, жалюзи",
  "5718": "Продажа каминов, экранов для каминов и аксессуаров",
  "5719": "Различные специализирован-ные магазины бытовых принадлежностей",
  "5722": "Другие магазины оборудования, домашние приборы",
  "5732": "Магазины электро-товаров",
  "5733": "Продажа музыкальных инструментов, фортепиано, нот",
  "5811": "Поставщики провизии",
  "5812": "Рестораны, места общественного питания",
  "5813": "Бары, коктейль-бары, дискотеки, ночные клубы и таверны - места продажи алкогольных напитков",
  "5814": "Рестораны быстрого обслуживания",
  "5912": "Аптеки, фармацевтические магазины",
  "5921": "Магазины с продажей спиртных напитков навынос (пиво, вино и ликер)",
  "5932": "Антикварные магазины - продажа, ремонт и услуги восстановления",
  "5940": "Веломагазины - продажа и обслуживание",
  "5941": "Магазины спортивных товаров",
  "5942": "Книжные магазины",
  "5943": "Магазины офисных, школьных принадлежностей, канцтоваров",
  "5944": "Магазины по продаже часов, ювелирных изделий и изделий из серебра",
  "5945": "Детские магазины",
  "5946": "Магазины по продаже камер, фильмов, видео и другого фотографического оборудования и приборов",
  "5947": "Магазины открыток, подарков, новинок и сувениров",
  "5948": "Магазины кожаных изделий, багажных сумок",
  "5949": "Магазины ткани, ниток рукоделия, шитья",
  "5950": "Магазины хрусталя и изделий из стекла",
  "5973": "Магазины религиозных товаров",
  "5976": "Ортопедические товары",
  "5977": "Магазины косметики",
  "5983": "Горючее топливо - уголь, нефть, разжиженный бензин, дрова",
  "5992": "Флористика",
  "5993": "Табачные магазины",
  "5994": "Дилеры по продаже печатной продукции",
  "5995": "Зоомагазины",
  "5999": "Различные магазины и специальные розничные магазины",
  "6211": "Ценные бумаги- брокеры/дилеры",
  "6300": "Продажа страхования, гарантированное размещение, премии",
  "6513": "Агенты и менеджеры по недвижимости — аренда",
  "7011": "Отели, сдаваемое жилье, не представленное в списке",
  "7032": "Спортивные и рекреационные лагеря",
  "7033": "Кемпинги и трейлерные парки",
  "7211": "Услуги прачечных",
  "7216": "Химчистки",
  "7221": "Фотостудии",
  "7230": "Парикмахерские, салоны красоты",
  "7251": "Ремонт обуви, чистка обуви и головных уборов",
  "7261": "Ритуальные услуги и крематории",
  "7297": "Массажная приемная. Терапевтические приемные, предлагающие услуги массажа.",
  "7298": "Салоны СПА-терапии",
  "7299": "Иной сервис",
  "7311": "Рекламные услуги",
  "7338": "Услуги копировальных центров",
  "7372": "Программирование, обработка данных, интегрированные системы, дизайн",
  "7379": "Ремонт и техническое обслуживание компьютерной техники",
  "7395": "Фотостудии, фотолаборатории",
  "7399": "Бизнес - сервис",
  "742": "Ветеринарные услуги",
  "7512": "Прокат автомобилей.",
  "7523": "Паркинги и гаражи",
  "7534": "Восстановление и ремонт покрышек",
  "7538": "Магазины автосервиса",
  "7542": "Автомойки",
  "7622": "Ремонт электроники",
  "7629": "Ремонт бытовой техники. ремонт электроприборы",
  "763": "Сельско-хозяйственные кооперативные общества",
  "7631": "Центры ремонта часов и чистки ювелирных изделий",
  "7699": "Общий ремонт",
  "7832": "Кинотеатры",
  "7922": "Агентства по продаже билетов",
  "7932": "Бильярд и Пул",
  "7941": "Спортивные поля, коммерческие спортивные состязания, профессиональные спортивные клубы, спортивные промоутеры",
  "7991": "Туристические аттракционы и шоу",
  "7996": "Луна-парки, карнавалы, цирки, предсказатели будущего",
  "7997": "Спортивные клубы и секции",
  "7999": "Спортивные развлечения",
  "8011": "Доктора - нигде ранее не классифицируемые",
  "8021": "Дантисты, ортодантисты",
  "8031": "Врачи узкой специализации (Остеопаты)",
  "8042": "Оптометристы, офтальмологи",
  "8043": "Оптика, оптические товары, и очки",
  "8050": "Уход и Средства Персонального обслуживания (санатории, роддома, дома пристарелых)",
  "8062": "Больницы",
  "8071": "Зубные и медицинские лаборатории",
  "8099": "Медицинские услуги, не представленные в списке",
  "8111": "Адвокаты, юридические услуги",
  "8211": "Школы, начальная и средняя",
  "8220": "Колледжи, университеты, профессиональные школы, и младшие колледжи",
  "8241": "Школы, корреспонденция",
  "8249": "Школы, торговые и профессионально-технические",
  "8299": "Услуги ухода за детьми",
  "8398": "Организации благотворительные и общественные службы",
  "8661": "Организации, религиозные",
  "8699": "Организации, членство - нигде ранее не классифицируемые",
  "8911": "Архитектурные, инжиниринговые услуги и услуги рассмотрения",
  "8931": "Бухгалтерский учет, ревизия, и бухгалтерские услуги"
}

drp_mcc_schema = 'sbx_da'
drp_mcc_table = 'tmp_shestopalov_acq_mcc_month'
drp_superset_grant_role = 'raisa_superset'
target_fq = f'{drp_mcc_schema}.{drp_mcc_table}'

print('dict size =', len(MCC_NAME_RU))
print('target =', target_fq)


In [ ]:
def _norm_mcc(x) -> str:
    s = str(x).strip()
    if s.isdigit():
        return str(int(s))  # '0742' / '742' → '742'
    return s


def _lookup_name(mcc) -> str:
    k = _norm_mcc(mcc)
    return MCC_NAME_RU.get(k) or MCC_NAME_RU.get(str(mcc).strip()) or f'MCC {k}'


drp_user = input('DRP user: ').strip()
drp_password = getpass.getpass('DRP password: ')
drp = connect(
    to='DRP',
    user_params={'user_name': drp_user, 'password': drp_password},
)

with drp:
    raw = drp.fetch(f'SELECT * FROM {target_fq}')

print('loaded rows =', len(raw))
print('columns =', list(raw.columns))
display(raw.head(3))


In [ ]:
df = raw.copy()
df.columns = [str(c).strip().lower() for c in df.columns]
if 'mcc' not in df.columns:
    raise RuntimeError('В таблице нет колонки mcc')

df['mcc'] = df['mcc'].map(lambda x: None if pd.isna(x) else str(x).strip())
df['mcc_name_ru'] = df['mcc'].map(_lookup_name)
df['mcc_label'] = df.apply(
    lambda r: (str(r['mcc']) + ' — ' + str(r['mcc_name_ru'])) if r['mcc'] else r['mcc_name_ru'],
    axis=1,
)

unknown = df.loc[df['mcc_name_ru'].str.startswith('MCC ', na=False), 'mcc'].dropna().unique().tolist()
print('rows =', len(df))
print('unique mcc =', df['mcc'].nunique())
print('unknown / fallback labels =', len(unknown), unknown[:20])
display(
    df.groupby(['mcc', 'mcc_name_ru'], as_index=False)['trx_sum']
    .count()
    .rename(columns={'trx_sum': 'n_rows'})
    .head(15)
)


In [ ]:
# Перезапись DRP (DROP + CREATE + GRANT) — без Impala
upload_df = df.copy()
for c in upload_df.columns:
    upload_df[c] = upload_df[c].map(lambda x: None if pd.isna(x) else str(x)).astype(object)

col_defs = [f'"{str(c).replace(chr(34), chr(34)+chr(34))}" TEXT' for c in upload_df.columns]
create_sql = f'CREATE TABLE {target_fq} (\n  ' + ',\n  '.join(col_defs) + '\n)'

with drp:
    drp.execute(f'DROP TABLE IF EXISTS {target_fq}')
    drp.execute(create_sql)
    drp.write(table=target_fq, df=upload_df, mode='append')
    cnt_df = drp.fetch(f'select count(*) as row_cnt from {target_fq}')
    try:
        drp.execute(f'GRANT USAGE ON SCHEMA {drp_mcc_schema} TO {drp_superset_grant_role}')
        drp.execute(f'GRANT SELECT ON TABLE {target_fq} TO {drp_superset_grant_role}')
        print(f'OK: GRANT SELECT ON {target_fq} TO {drp_superset_grant_role}')
    except Exception as grant_exc:
        print('WARNING: GRANT failed:', type(grant_exc).__name__, str(grant_exc)[:400])
        print(f'  GRANT SELECT ON TABLE {target_fq} TO {drp_superset_grant_role};')

print('OK DRP rows =', int(pd.to_numeric(cnt_df.iloc[0, 0], errors='coerce')))
print('Superset: Sync columns → dimension mcc_label (или mcc_name_ru)')
